## Modelo 6 - BERT + Neural Network

La idea es usar todas las variables que veniamos utilizando antes pero también el titulo + descripción. Para transformar esto a variables numericas vamos a utilizar un sentence transformer para obtener los embeddings. Luego la cabeza de regresión del modelo será un red neuronal.

### Leemos los datos

In [1]:
import pandas as pd

train = pd.read_csv("datos_entrenamiento.csv")
valid = pd.read_csv("datos_validacion.csv")

### Construimos el texto

Generamos un campo donde tenemos lo siguiente:

Title: ....

Description: ...


In [2]:
train["texto"] = (
    "Title: " +
    train["title"].fillna("") +
    ". Description: " +
    train["description"].fillna("")
)

valid["texto"] = (
    "Title: " +
    valid["title"].fillna("") +
    ". Description: " +
    valid["description"].fillna("")
)

In [3]:
import sys
!{sys.executable} -m pip install sentence-transformers

     ---------------------------------------- 0.0/596.7 kB ? eta -:--:--
     ----------- -------------------------- 174.1/596.7 kB 5.1 MB/s eta 0:00:01
     -------------------------------------  593.9/596.7 kB 7.4 MB/s eta 0:00:01
     -------------------------------------- 596.7/596.7 kB 6.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/771.9 kB ? eta -:--:--
     ---------------------------- -------- 604.2/771.9 kB 12.6 MB/s eta 0:00:01
     -------------------------------------- 771.9/771.9 kB 9.8 MB/s eta 0:00:00
     ---------------------------------------- 0.0/122.0 MB ? eta -:--:--
     --------------------------------------- 0.6/122.0 MB 17.9 MB/s eta 0:00:07
     --------------------------------------- 1.2/122.0 MB 14.5 MB/s eta 0:00:09
      -------------------------------------- 1.7/122.0 MB 12.3 MB/s eta 0:00:10
      -------------------------------------- 2.3/122.0 MB 11.4 MB/s eta 0:00:11
      -------------------------------------- 2.9/122.0 MB 12.


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Obtenemos los embeddings

In [4]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("all-MiniLM-L6-v2")

X_text_train = modelo.encode(
    train["texto"].tolist(),
    show_progress_bar=True
)

X_text_valid = modelo.encode(
    valid["texto"].tolist(),
    show_progress_bar=True
)

c:\Users\Sebastian\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Sebastian\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sebastian\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer M

### Preparamos las otras variables

In [7]:
import numpy as np

variables_numericas = [
    "release_year",
    "runtime",
    "seasons",
    "log_imdb_votes",
    "votos_faltantes",
    "cantidad_generos",
    "cantidad_actores"
]

variables_categoricas = [
    "type",
    "age_certification",
    "genero_principal"
]

Pasamos las variables categoricas a numericas con OneHotEncoder

In [8]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_cat_train = encoder.fit_transform(
    train[variables_categoricas]
)

X_cat_valid = encoder.transform(
    valid[variables_categoricas]
)

Normalizamos las variables númericas

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_num_train = scaler.fit_transform(
    train[variables_numericas]
)

X_num_valid = scaler.transform(
    valid[variables_numericas]
)

Concatenamos todas las variables: Text, Numericas y Categóricas

In [10]:
X_train = np.concatenate(
    [
        X_text_train,
        X_num_train,
        X_cat_train
    ],
    axis=1
)

X_valid = np.concatenate(
    [
        X_text_valid,
        X_num_valid,
        X_cat_valid
    ],
    axis=1
)

In [11]:
print(X_train.shape)
print(X_valid.shape)

(2904, 424)
(727, 424)


### Variable Respuesta

In [12]:
y_train = train["imdb_score"].values
y_valid = valid["imdb_score"].values

### Pasamos a tensores

Este paso es para poder implementar la red neuronal

In [13]:
import torch

X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_valid = torch.tensor(
    X_valid,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
).reshape(-1,1)

y_valid = torch.tensor(
    y_valid,
    dtype=torch.float32
).reshape(-1,1)

### Implementación de la Red Neuronal

In [14]:
import torch
import torch.nn as nn

class MLPRegresion(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.modelo = nn.Sequential(

            nn.Linear(input_dim, 128),

            nn.ReLU(),

            nn.Dropout(0.30),

            nn.Linear(128, 64),

            nn.ReLU(),

            nn.Dropout(0.30),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        return self.modelo(x)

Creamos la red

In [15]:
input_dim = X_train.shape[1]

modelo = MLPRegresion(input_dim)

### Función de pérdida y Optimizador

Como estamos usando MSE implementamos eso. Y usamos Adam como optimizador

In [17]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    modelo.parameters(),
    lr=1e-3
)

### Dataloader

Esto CREO que es para entrenar por batches en vez de usar cada fila para realizar una actualizacion de los pesos

In [18]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

### Entrenamiento

In [25]:
num_epochs = 500

for epoch in range(num_epochs):

    modelo.train()

    loss_total = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = modelo(X_batch)

        loss = criterion(pred, y_batch)

        loss.backward()

        optimizer.step()

        loss_total += loss.item()

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch+1}: "
            f"{loss_total/len(train_loader):.4f}"
        )

Epoch 10: 0.0584
Epoch 20: 0.0576
Epoch 30: 0.0654
Epoch 40: 0.0604
Epoch 50: 0.0598
Epoch 60: 0.0595
Epoch 70: 0.0643
Epoch 80: 0.0619
Epoch 90: 0.0598
Epoch 100: 0.0627
Epoch 110: 0.0672
Epoch 120: 0.0585
Epoch 130: 0.0623
Epoch 140: 0.0597
Epoch 150: 0.0632
Epoch 160: 0.0638
Epoch 170: 0.0613
Epoch 180: 0.0578
Epoch 190: 0.0611
Epoch 200: 0.0564
Epoch 210: 0.0601
Epoch 220: 0.0599
Epoch 230: 0.0653
Epoch 240: 0.0574
Epoch 250: 0.0538
Epoch 260: 0.0631
Epoch 270: 0.0596
Epoch 280: 0.0626
Epoch 290: 0.0642
Epoch 300: 0.0599
Epoch 310: 0.0571
Epoch 320: 0.0529
Epoch 330: 0.0645
Epoch 340: 0.0597
Epoch 350: 0.0586
Epoch 360: 0.0593
Epoch 370: 0.0587
Epoch 380: 0.0630
Epoch 390: 0.0580
Epoch 400: 0.0566
Epoch 410: 0.0625
Epoch 420: 0.0612
Epoch 430: 0.0618
Epoch 440: 0.0547
Epoch 450: 0.0608
Epoch 460: 0.0605
Epoch 470: 0.0600
Epoch 480: 0.0529
Epoch 490: 0.0587
Epoch 500: 0.0569


### Predicciones

In [26]:
modelo.eval()

with torch.no_grad():

    predicciones = modelo(X_valid)

mse = torch.mean(
    (predicciones - y_valid) ** 2
)

print("MSE:", mse.item())

MSE: 0.9558727741241455
